In [ ]:
!pip install google-adk==1.28.0

In [ ]:
import os
from getpass import getpass

# Prompt the user for their API key securely
api_key = getpass('Enter your Gemini API Key: ')

# Set the API key as an environment variable for ADK to use
os.environ['GEMINI_API_KEY'] = api_key

print("✅ API Key configured successfully! Let the fun begin.")

Enter your Gemini API Key: ··········
✅ API Key configured successfully! Let the fun begin.


## ⚙️ 1. The Overall Architecture & Setup

This module demonstrates how to build an advanced, multi-skilled AI agent using the Google Agent Development Kit (ADK). We are moving away from hardcoding rigid prompts and instead teaching the agent how to dynamically load capabilities on demand.

Our architecture is broken down into four distinct phases:
1. **The Execution Engine:** We set up an `InMemorySessionService` (the agent's short-term memory) and a helper function to run queries.
2. **Skill Definition (In-Memory):** We define skills directly inside Python using the `models.Skill` class, making the code highly portable.
3. **The Toolset Assembly:** We bundle individual skills into a single `SkillToolset` and hand it to our agent.
4. **Automatic Routing (The Execution):** The agent automatically reads the user's prompt and decides which skill it needs to pull from the toolset to solve the problem.

Let's start by setting up our memory and execution engine.

In [ ]:
import asyncio
from IPython.display import display, Markdown
from google.genai.types import Content, Part
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.adk.agents import Agent

# --- 1. Initialize Memory & Execution Engine ---
# The Session Service manages conversation history and state in memory.
session_service = InMemorySessionService()
user_id = "adk_course_student_001"

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str):
    """Executes a user query against the agent and displays the response."""
    print(f"\n🚀 Running query for agent: '{agent.name}'...")
    runner = Runner(agent=agent, session_service=session_service, app_name=agent.name)
    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    print("-" * 50)
    print("✅ Final Response:")
    display(Markdown(final_response))
    print("-" * 50 + "\n")
    return final_response

## 🤝 2. Breakdown: The Vendor Negotiation Skill

This first skill turns the agent into a tough but fair financial negotiator. It is structured using the L1/L2/L3 "progressive disclosure" architecture to keep the context window lightweight.

* **L1 (Frontmatter/Metadata):** The `description` acts as the "trigger prompt." When the user asks about budgets or contracts, this tells the agent, *"Hey, use this skill!"*
* **L2 (Instructions):** This is the core logic payload. It tells the agent the exact steps it must take: read the tactics, use the email template, and aim to lower the price.
* **L3 (Resources):** * `references`: Contains a virtual file (`negotiation_tactics.md`) listing approved company strategies.
    * `assets`: Contains a virtual file (`email_template.txt`) enforcing a strict structure for the output.

In [ ]:
from google.adk.skills import models

# --- 2. Define the Vendor Negotiator Skill ---
negotiation_skill = models.Skill(
    # L1 (Metadata): Loaded at startup for skill discovery.
    frontmatter=models.Frontmatter(
        name="vendor-negotiation-skill",
        description="Extracts pricing strategies. Use this skill when negotiating event vendor contracts, budgets, or pricing.",
    ),

    # L2 (Instructions): The primary instructions.
    instructions=(
        "You are a tough but fair negotiator. When asked to negotiate with a vendor:\n"
        "1. Read 'references/negotiation_tactics.md' for approved strategies.\n"
        "2. Draft a response using the template in 'assets/email_template.txt'.\n"
        "3. Always aim to get the vendor's quote down to match our strict budget."
    ),

    # L3 (Resources): Loaded on-demand by the agent when referenced in L2.
    resources=models.Resources(
        references={
            "negotiation_tactics.md": (
                "# Approved Negotiation Tactics\n\n"
                "**1. The 'Anchor':** Always acknowledge their price respectfully, but immediately anchor the conversation to our strict, non-negotiable budget constraints.\n\n"
                "**2. The 'Value Add' (CRITICAL PIVOT):** If the vendor absolutely refuses to lower their base price, you MUST pivot to this strategy. Ask for complimentary upgrades to increase the overall ROI. Examples: free premium lighting, an extra hour of service, or waived delivery/setup fees.\n\n"
                "**3. The 'Walk Away':** If the vendor refuses to lower their base price, politely state that we are currently evaluating two other highly competitive quotes that fit our budget, and we are prepared to move forward with them if an agreement cannot be reached."
            )
        },
        assets={
            "email_template.txt": (
                "Subject: Regarding your quote for our upcoming Event\n\n"
                "Hi [Vendor Name],\n\n"
                "Thank you for the quote of $[Quote Amount]. However, our hard cap is $[Budget Amount].\n"
                "[Insert ONE chosen strategy from the references here]\n\n"
                "Looking forward to your thoughts,\n"
                "The Autonomous Event Team"
            )
        }
    )
)

print("✅ Vendor Negotiation Skill defined in memory!")

✅ Vendor Negotiation Skill defined in memory!


## 🚨 3. Breakdown: The Crisis Management Skill

This second skill completely shifts the agent's persona to handle sudden event emergencies calmly and professionally.

* **L1 (Frontmatter/Metadata):** Tells the agent to activate this tool if the user mentions weather issues, cancellations, or emergencies.
* **L2 (Instructions):** The core payload tells the agent to act as a calm crisis manager, check emergency protocols, and draft an action plan.
* **L3 (Resources):** Contains `emergency_protocols.md`, a virtual document that tells the agent exactly what to do in specific scenarios (e.g., if it rains, move to Backup Hall B).

We will define this skill, bundle both skills into a `SkillToolset`, and equip our `super_agent`.

In [ ]:
from google.adk.tools import skill_toolset

# --- 3. Define a Second Skill (Crisis Management) ---
crisis_skill = models.Skill(
    frontmatter=models.Frontmatter(
        name="crisis-management-skill",
        description="Use this skill for handling sudden event emergencies, weather issues, or vendor cancellations.",
    ),
    instructions=(
        "You are a calm crisis manager. When an emergency strikes:\n"
        "1. Consult 'references/emergency_protocols.md' for the contingency plan.\n"
        "2. Draft an immediate action plan for the team."
    ),
    resources=models.Resources(
        references={
            "emergency_protocols.md": (
                "# Emergency Protocols\n"
                "- **Weather:** Move outdoor activities to Backup Hall B. Issue push notifications.\n"
                "- **Vendor No-Show:** Contact secondary vendors. Deploy backup catering."
            )
        }
    )
)

print("✅ Crisis Management Skill defined in memory!")

✅ Crisis Management Skill defined in memory!


In [ ]:
# --- 4. Load BOTH Skills into a single Toolset ---
multi_skill_toolset = skill_toolset.SkillToolset(skills=[negotiation_skill, crisis_skill])

# --- 5. Create a Multi-Skilled Agent ---
super_agent = Agent(
    name="super_event_agent",
    model="gemini-2.5-flash",
    instruction="You are a highly capable event manager. Use your available skills to resolve the user's situation appropriately.",
    tools=[multi_skill_toolset]
)

print("🤖 Super Agent equipped with multiple skills is ready to route automatically!")

🤖 Super Agent equipped with multiple skills is ready to route automatically!


## 💡 4. The Core Takeaway: Preventing Context Bloat



By defining the skills this way, we perfectly illustrate how to prevent "context bloat."

The agent doesn't have to memorize the `email_template.txt` or the `emergency_protocols.md` all at once. It simply looks at the L1 Metadata menu for both skills, picks the right tool for the job based on the user's prompt, and *then* loads the necessary L2 and L3 data to execute the task.

Let's run two distinct test queries and watch the agent automatically route them to the correct skill!

In [ ]:
async def run_automatic_routing_tests():
    session = await session_service.create_session(
        app_name=super_agent.name,
        user_id=user_id
    )

    # Test 1: The prompt context will cause the agent to read the L1 metadata
    # and automatically load the L2/L3 payload of the Crisis Management skill.
    query1 = "URGENT! A massive thunderstorm just hit and the outdoor venue is flooded. What do we do?!"
    print(f"\n🔴 TEST 1: The Emergency")
    print(f"🗣️ User: {query1}")
    await run_agent_query(super_agent, query1, session, user_id)

    # Test 2: Automatically triggers the Vendor Negotiation skill.
    query2 = (
        "The caterer, 'Delicious Eats', just replied to us. They stated that their "
        "quote of $5000 is absolutely firm and they refuse to lower the base price "
        "by a single cent. Our strict budget is only $3500. We need to push back. "
        "Draft a response to them."
    )
    print(f"\n🔵 TEST 2: The Hardball Negotiation")
    await run_agent_query(super_agent, query2, session, user_id)

# Execute the test
await run_automatic_routing_tests()


🔴 TEST 1: The Emergency
🗣️ User: URGENT! A massive thunderstorm just hit and the outdoor venue is flooded. What do we do?!

🚀 Running query for agent: 'super_event_agent'...
--------------------------------------------------
✅ Final Response:


Okay team, here's the immediate action plan for the flooded outdoor venue:

**Immediate Action Plan:**

1.  **Relocation:** All outdoor activities are to be moved to **Backup Hall B** immediately.
2.  **Communication:** Issue push notifications to all attendees informing them of the venue change to Backup Hall B due to the thunderstorm and flooding.

Please execute these steps with urgency and report once completed.

--------------------------------------------------


🔵 TEST 2: The Hardball Negotiation

🚀 Running query for agent: 'super_event_agent'...
--------------------------------------------------
✅ Final Response:


Subject: Regarding your quote for our upcoming Event

Hi Delicious Eats,

Thank you for the quote of $5000. However, our hard cap is $3500.

We understand that your base price is firm. To help us bridge this gap and proceed with Delicious Eats, would you be open to offering some complimentary value-adds? We are particularly interested in options such as an extra hour of service, inclusion of premium non-alcoholic beverages, or a waiver of delivery and setup fees.

We are currently evaluating a couple of other highly competitive quotes that align with our budget, and we would ideally prefer to work with your team.

Looking forward to your thoughts,
The Autonomous Event Team

--------------------------------------------------

